# Benchmark analysis

Reads `results/results.csv` and emits the five charts that ship with the Medium article into `article/charts/`.

Charts are deliberately plain — readable beats pretty for a Medium embed.

In [ ]:
import csv
from pathlib import Path
from statistics import median

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'harness' else Path.cwd()
RESULTS_CSV = REPO_ROOT / 'results' / 'results.csv'
CHARTS_DIR = REPO_ROOT / 'article' / 'charts'
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

WORKFLOW_LABEL = {'a': 'A — Plan Mode', 'b': 'B — Spec Kit', 'c': 'C — BMAD'}
TASK_LABEL = {1: 'T1 trivial', 2: 'T2 medium', 3: 'T3 ambiguous', 4: 'T4 brownfield'}

df = pd.read_csv(RESULTS_CSV)
df['workflow_label'] = df['workflow'].map(WORKFLOW_LABEL)
df['task_label'] = df['task'].map(TASK_LABEL)
df

## Helper

In [ ]:
def grouped_bar(df, metric, ylabel, title, filename, agg='mean'):
    tasks = sorted(df['task'].unique())
    workflows = sorted(df['workflow'].unique())
    n_workflows = len(workflows)
    width = 0.8 / n_workflows

    fig, ax = plt.subplots(figsize=(8, 4.5))
    for i, wf in enumerate(workflows):
        means, errs = [], []
        for t in tasks:
            cell = df[(df['workflow'] == wf) & (df['task'] == t)][metric]
            if len(cell) == 0:
                means.append(0)
                errs.append(0)
            else:
                means.append(cell.mean() if agg == 'mean' else cell.median())
                errs.append(cell.std() if len(cell) > 1 else 0)
        xs = [t - 0.4 + width * (i + 0.5) for t in tasks]
        ax.bar(xs, means, width=width, yerr=errs, capsize=4, label=WORKFLOW_LABEL[wf])

    ax.set_xticks(tasks)
    ax.set_xticklabels([TASK_LABEL[t] for t in tasks])
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    fig.tight_layout()
    fig.savefig(CHARTS_DIR / filename, dpi=150)
    plt.show()
    return fig

## Wall-clock time

In [ ]:
grouped_bar(
    df, 'duration_seconds',
    'Seconds', 'Wall-clock time per task (mean ± std, n=3)',
    'time_per_task.png',
)

## Cost

In [ ]:
grouped_bar(
    df, 'cost_usd',
    'USD', 'Cost per task (mean ± std, n=3)',
    'cost_per_task.png',
)

## Lines changed (proxy for over-engineering)

In [ ]:
df['lines_changed'] = df['lines_added'] + df['lines_removed']
grouped_bar(
    df, 'lines_changed',
    'Lines added + removed', 'Lines changed per task (mean ± std, n=3)',
    'lines_per_task.png',
)

## Subjective score — the headline chart

In [ ]:
ax = grouped_bar(
    df, 'subjective_score',
    'Score (1–5)', 'Subjective score per task (mean ± std, n=3)',
    'score_per_task.png',
).axes[0]
ax.set_ylim(0, 5.5)

## Summary table

In [ ]:
summary = (
    df.groupby('workflow_label')
      .agg(
          median_time_s=('duration_seconds', 'median'),
          median_cost_usd=('cost_usd', 'median'),
          median_score=('subjective_score', 'median'),
          median_lines=('lines_changed', 'median'),
          runs=('subjective_score', 'count'),
      )
      .round(2)
)
summary.to_csv(CHARTS_DIR / 'summary_table.csv')
summary